I read through the algebraic equations related to ECDSA, and I can understand it, but I still don't get it conceptually. In this post, I'm capturing the process of convincing myself that you can prove possession of a secret, without revealing the secret itself.

I got a few realisations from Adam Gibson’s *From Zero (Knowledge) to Bulletproofs*.

*Section 1.3, p. 4*

> In isolation, it's not really useful for Alice to prove to Bob that she knows, say, the vectors $\mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3$ to which the commitments $C_1, C_2, C_3$ committed, if she doesn't at some later point reveal them (more about “commitments” in Section 2, if you're new to them).
>
> But this can be part of a larger proof: for example, <mark>Alice may prove that $C_4$ is a commitment to a scalar value $t$, which is the dot product $t = \mathbf{v}_1 \cdot \mathbf{v}_2$; still without revealing any of these values.</mark>

*Section 2.2, pp. 5–6*

> Commitments are only possible because of the existence of one-way functions; you need to be able to produce an output that doesn’t, on its own, reveal the input. Cryptographic hash functions (like SHA256) perform this role perfectly. If I want to commit to the value “2”, I can send you its hash:
>
> `53c234e5e8472b6ac51c1ae1cab3fe06fad053beb8ebfd8977b010655bfdd3c3`
>
> This wouldn’t be too smart though; it would mean whenever I want to send you a commitment to the value “2”, I would always be sending 53....c3 - which wouldn’t hide the value very well! This is a lack of what’s called “semantic security”. However, it’s pretty easy to address this.
>
> Instead of committing to “2”, I commit to “2”+some random data, like “2xyzabc”. […] Because SHA256 has the property that you can’t generate “collisions” (can’t create two inputs with the same output), this still gives us the same essential security feature: once I’ve made the commitment, I can’t change my mind on what the value is later - and you can ask me to reveal it at a later stage of a protocol/system.
>
> Notice that by <mark>combining a hash function with an extra piece of random data</mark>, we have achieved what are known as the two key properties of any commitment scheme:
>
> - <mark>Hiding - a commitment $C$ does not reveal the value it commits to.</mark>
> - <mark>Binding - having made the commitment $C(m)$ to $m$, you can’t change your mind and open it as a commitment to a different message $m'$.</mark>

*Section 2.2, p. 6*

> So, a hash function provides the equipment to make commitments. However, what hash functions certainly don’t have is any kind of homomorphism. <mark>There is certainly no rule like $\mathrm{SHA256}(a) + \mathrm{SHA256}(b)$ $= \mathrm{SHA256}(a + b)$.</mark> So this is no good for our purposes, because we are going to need a *homomorphic commitment scheme* for the techniques we're going to use.

*Section 2.3, p. 6*

> Instead of using a hash function, we will use a point on an elliptic curve to achieve the same goal; the one way function here is scalar multiplication of curve points:
>
> $$C = rH + aG$$
>
> Here, $C$ is the curve point we will use as a commitment (and give to some counterparty), $a$ is the value we commit to (assume from now on it's a number, not a string), $r$ is the randomness which provides hiding, $G$ is as already mentioned the publically agreed generator of the elliptic curve, and $H$ is another curve point, for which **nobody** knows the discrete logarithm $q$ s.t. $H = qG$. This unknowness is vital, as we'll expand upon next.
>
> But, crucially for the rest of this document, this new commitment scheme *does* have a homomorphism:
>
> $$\begin{aligned} &C(r_1, a_1) + C(r_2, a_2) \\ &\quad = r_1 H + a_1 G + r_2 H + a_2 G \\ &\quad = (r_1 + r_2)H + (a_1 + a_2)G \\ &\quad = C(r_1 + r_2, a_1 + a_2) \end{aligned}$$
>
> In words: “<mark>the sum of the commitments to $a_1$ and $a_2$ is equal to a commitment to $(a_1 + a_2)$, as long as you choose the randomness for each of the first two commitments so that their sum is equal to randomness for the third.</mark>”